# Analyticx AI — QLoRA Fine-Tuning on Google Colab

Runtime -> Change runtime type -> T4 GPU
Then run cells top to bottom.

In [ ]:
!pip install -q transformers==4.45.2 peft==0.12.0 trl==0.11.4 datasets==3.0.0 accelerate==1.0.0 bitsandbytes==0.43.3

In [ ]:
from google.colab import files
uploaded = files.upload()  # upload train.jsonl and valid.jsonl

In [ ]:
import torch
from datasets import load_dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import SFTTrainer, SFTConfig

BASE_MODEL = 'Qwen/Qwen2.5-3B-Instruct'

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=torch.bfloat16)
tok = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=bnb, device_map='auto', trust_remote_code=True)
model = prepare_model_for_kbit_training(model)

lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias='none', task_type='CAUSAL_LM',
                  target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'])
model = get_peft_model(model, lora)

train_ds = load_dataset('json', data_files='train.jsonl', split='train')
valid_ds = load_dataset('json', data_files='valid.jsonl', split='train')

args = SFTConfig(output_dir='./checkpoints', num_train_epochs=3,
                 per_device_train_batch_size=2, gradient_accumulation_steps=8,
                 learning_rate=2e-4, warmup_ratio=0.03, lr_scheduler_type='cosine',
                 logging_steps=10, save_steps=100, eval_steps=100, save_total_limit=2,
                 bf16=True, optim='paged_adamw_8bit', max_grad_norm=0.3, max_seq_length=1024, report_to='none')

trainer = SFTTrainer(model=model, args=args, train_dataset=train_ds, eval_dataset=valid_ds, tokenizer=tok)
trainer.train()
trainer.save_model('./adapter')
tok.save_pretrained('./adapter')

In [ ]:
!zip -r adapter.zip adapter
from google.colab import files
files.download('adapter.zip')